# `02-data-cleaning-and-exploration.ipynb`: Cleaning the scraped Arrests and Arrests & Incidents Logs and doing some preliminary data exploration

**Written by**: Diego Torrealba

**Project**: umpd-crime

**Date**: 2026-11-07

## Setup

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path

BASE_DIR = Path.cwd().parent
RAW_DATA_DIR = BASE_DIR / "data" / "raw" 
CURRENT_YEAR = time.localtime().tm_year
CURRENT_DAY = pd.to_datetime(f'{CURRENT_YEAR}-{time.localtime().tm_mon}-{time.localtime().tm_mday}')

# for time of day categorization
## 0 = 12 am; 23:59 - 11:59 pm
TOD_CATEGORIES = {
  'bins' : [0, 5, 11, 17, 23, 24],
  'labels' : ['night', 'morning', 'afternoon', 'evening', 'night']
}


# Semester start and end dates
SEMESTER_RANGES = {
  # 2017-2018
  'W_2018' : {'start_date': pd.to_datetime('2018-01-02'), 'end_date': pd.to_datetime('2018-01-22')},
  'SP_2018' : {'start_date': pd.to_datetime('2018-01-24'), 'end_date': pd.to_datetime('2018-05-18')},
  'SU_2018' : {'start_date': pd.to_datetime('2018-05-29'), 'end_date': pd.to_datetime('2018-08-17')},
  
  # 2018-2019
  'F_2018' : {'start_date': pd.to_datetime('2018-08-27'), 'end_date': pd.to_datetime('2018-12-18')},
  'W_2019' : {'start_date': pd.to_datetime('2019-01-02'), 'end_date': pd.to_datetime('2019-01-22')},
  'SP_2019' : {'start_date': pd.to_datetime('2019-01-28'), 'end_date': pd.to_datetime('2019-05-22')},
  'SU_2019' : {'start_date': pd.to_datetime('2019-05-28'), 'end_date': pd.to_datetime('2019-08-16')},
  
  # 2019-2020
  'F_2019' : {'start_date': pd.to_datetime('2019-08-26'), 'end_date': pd.to_datetime('2019-12-17')},
  'W_2020' : {'start_date': pd.to_datetime('2020-01-02'), 'end_date': pd.to_datetime('2020-01-22')},
  'SP_2020' : {'start_date': pd.to_datetime('2020-01-27'), 'end_date': pd.to_datetime('2020-05-20')},
  'SU_2020' : {'start_date': pd.to_datetime('2020-06-01'), 'end_date': pd.to_datetime('2020-08-21')},
  
  # 2020-2021
  'F_2020' : {'start_date': pd.to_datetime('2020-08-31'), 'end_date': pd.to_datetime('2020-12-22')},
  'W_2021' : {'start_date': pd.to_datetime('2021-01-04'), 'end_date': pd.to_datetime('2021-01-22')},
  'SP_2021' : {'start_date': pd.to_datetime('2021-01-25'), 'end_date': pd.to_datetime('2021-05-19')},
  'SU_2021' : {'start_date': pd.to_datetime('2021-06-01'), 'end_date': pd.to_datetime('2021-08-20')},
  
  # 2021-2022
  'F_2021' : {'start_date': pd.to_datetime('2021-08-30'), 'end_date': pd.to_datetime('2021-12-21')},
  'W_2022' : {'start_date': pd.to_datetime('2022-01-03'), 'end_date': pd.to_datetime('2022-01-21')},
  'SP_2022' : {'start_date': pd.to_datetime('2022-01-24'), 'end_date': pd.to_datetime('2022-05-18')},
  'SU_2022' : {'start_date': pd.to_datetime('2022-05-31'), 'end_date': pd.to_datetime('2022-08-19')},
  
  # 2022-2023
  'F_2022' : {'start_date': pd.to_datetime('2022-08-29'), 'end_date': pd.to_datetime('2022-12-20')},
  'W_2023' : {'start_date': pd.to_datetime('2023-01-03'), 'end_date': pd.to_datetime('2023-01-23')},
  'SP_2023' : {'start_date': pd.to_datetime('2023-01-25'), 'end_date': pd.to_datetime('2023-05-19')},
  'SU_2023' : {'start_date': pd.to_datetime('2023-05-30'), 'end_date': pd.to_datetime('2023-08-18')},
  
  # 2023-2024
  'F_2023' : {'start_date': pd.to_datetime('2023-08-28'), 'end_date': pd.to_datetime('2023-12-19')},
  'W_2024' : {'start_date': pd.to_datetime('2024-01-02'), 'end_date': pd.to_datetime('2024-01-22')},
  'SP_2024' : {'start_date': pd.to_datetime('2024-01-24'), 'end_date': pd.to_datetime('2024-05-17')},
  'SU_2024' : {'start_date': pd.to_datetime('2024-05-28'), 'end_date': pd.to_datetime('2024-08-16')},
  
  # 2024-2025
  'F_2024' : {'start_date': pd.to_datetime('2024-08-26'), 'end_date': pd.to_datetime('2024-12-17')},
  'W_2025' : {'start_date': pd.to_datetime('2025-01-02'), 'end_date': pd.to_datetime('2025-01-22')},
  'SP_2025' : {'start_date': pd.to_datetime('2025-01-27'), 'end_date': pd.to_datetime('2025-05-21')},
  'SU_2025' : {'start_date': pd.to_datetime('2025-06-02'), 'end_date': pd.to_datetime('2025-08-22')},
  
  # 2025-2026
  'F_2025' : {'start_date': pd.to_datetime('2025-09-02'), 'end_date': pd.to_datetime('2025-12-20')},
  'W_2026' : {'start_date': pd.to_datetime('2026-01-05'), 'end_date': pd.to_datetime('2026-01-23')},
  'SP_2026' : {'start_date': pd.to_datetime('2026-01-26'), 'end_date': pd.to_datetime('2026-05-18')},
  'SU_2026' : {'start_date': pd.to_datetime('2026-06-01'), 'end_date': pd.to_datetime('2026-08-21')},
  
  # 2026-now
  'F_2026': {'start_date': pd.to_datetime('2026-08-31'), 'end_date': CURRENT_DAY}
}

## SEMESTER_CONDITIONS
SEMESTER_LABELS = ['Winter 2018', 'Spring 2018', 'Summer 2018', 'Fall 2018', 'Winter 2019', 'Spring 2019', 'Summer 2019', 'Fall 2019', 'Winter 2020', 'Spring 2020', 'Summer 2020', 'Fall 2020', 'Winter 2021', 'Spring 2021', 'Summer 2021', 'Fall 2021', 'Winter 2022', 'Spring 2022', 'Summer 2022', 'Fall 2022', 'Winter 2023', 'Spring 2023', 'Summer 2023', 'Fall 2023', 'Winter 2024', 'Spring 2024', 'Summer 2024', 'Fall 2024', 'Winter 2025', 'Spring 2025', 'Summer 2025', 'Fall 2025', 'Winter 2026', 'Spring 2026', 'Summer 2026', 'Fall 2026']

def get_semester_conditions(series):
  """Returns list of Boolean conditions to set semester time frames, written by GPT-6.1 Sol on 2026-11-06 (stumped on dynamic condition creation)"""
  
  dates = pd.to_datetime(series).dt.normalize()
  
  return [
    dates.between(
      semester['start_date'],
      semester['end_date'],
      inclusive="both"
    )
    for semester in SEMESTER_RANGES.values()
  ]

arrests_df = pd.read_csv(RAW_DATA_DIR / "arrest-ledger" / f"umpd_arrest_ledger_2018-{CURRENT_YEAR}.csv")
dci_df = pd.read_csv(RAW_DATA_DIR / 'daily-crime-incident-logs' / f'daily_crime_incident_log_2019-{CURRENT_YEAR}.csv')
arrests_merged_df = pd.read_csv(RAW_DATA_DIR.parent / "processed" / f"merged_arrest_ci_data_2018-{CURRENT_YEAR}.csv")

In [49]:
CURRENT_DAY

Timestamp('2026-11-07 00:00:00')

## Arrests Log

In [50]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3728 entries, 0 to 3727
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   arrest_number     3728 non-null   int64  
 1   arrest_date       3728 non-null   str    
 2   umpd_case_number  3728 non-null   str    
 3   age               27 non-null     float64
 4   race              3650 non-null   str    
 5   sex               3651 non-null   str    
 6   arrested_charge   3728 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 204.0 KB


The columns that have values for every row are: `arrest_number`, `arrest_date`, `umpd_case_number` and `arrested_charge`. 

The columns that have missing values by order of how many they're missing: 

In [51]:
arrests_df.isna().sum()

arrest_number          0
arrest_date            0
umpd_case_number       0
age                 3701
race                  78
sex                   77
arrested_charge        0
dtype: int64

This checks out, as `age` is absent from most rows on the website. 

### Fixing datatypes

In [52]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3728 entries, 0 to 3727
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   arrest_number     3728 non-null   int64  
 1   arrest_date       3728 non-null   str    
 2   umpd_case_number  3728 non-null   str    
 3   age               27 non-null     float64
 4   race              3650 non-null   str    
 5   sex               3651 non-null   str    
 6   arrested_charge   3728 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 204.0 KB


- `arrest_number` should be a string, as its an identifier;
- `arrest_date` should be a datetime object

In [53]:
# Converting arrests_df columns to appropriate data types
arrests_df['arrest_number'] = arrests_df['arrest_number'].astype(str)
arrests_df['arrest_date'] = pd.to_datetime(arrests_df['arrest_date'])

### Creating supplementary columns from given columns

In [54]:
arrests_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3728 entries, 0 to 3727
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   arrest_number     3728 non-null   str           
 1   arrest_date       3728 non-null   datetime64[us]
 2   umpd_case_number  3728 non-null   str           
 3   age               27 non-null     float64       
 4   race              3650 non-null   str           
 5   sex               3651 non-null   str           
 6   arrested_charge   3728 non-null   str           
dtypes: datetime64[us](1), float64(1), str(5)
memory usage: 204.0 KB


In [58]:
arrests_df['time_of_day'] = pd.cut(
  arrests_df['arrest_date'].dt.hour,
  bins=TOD_CATEGORIES['bins'],
  labels=TOD_CATEGORIES['labels'],
  ordered=False,
  include_lowest=True
)
arrests_df[['arrest_date', 'time_of_day']].sample(10)

,arrest_date,time_of_day
3642,2026-08-30 05:46:00,night
310,2018-10-24 13:10:00,afternoon
1264,2020-09-06 01:16:00,night
3346,2026-02-22 03:36:00,night
95,2018-03-19 14:10:00,afternoon
904,2019-09-21 23:41:00,evening
1988,2022-09-04 01:23:00,night
1203,2020-05-07 09:41:00,morning
2157,2023-03-23 17:05:00,afternoon
1679,2021-10-22 22:18:00,evening


In [59]:
arrests_df['time_of_day'].value_counts(sort=True)

time_of_day
night        1961
evening       761
afternoon     734
morning       272
Name: count, dtype: int64

In [61]:
arrests_df[arrests_df['time_of_day'] == 'night']['arrest_date'].sample(10)

2609   2024-07-20 01:00:00
2694   2024-09-20 01:43:00
3432   2026-04-19 01:14:00
3616   2026-08-16 00:38:00
2122   2023-02-14 00:54:00
1676   2021-10-22 00:09:00
905    2019-09-22 01:15:00
1534   2021-06-16 00:38:00
1506   2021-05-18 01:21:00
3575   2026-08-01 01:00:00
Name: arrest_date, dtype: datetime64[us]

In [78]:
arrests_df_sem_conditions = get_semester_conditions(arrests_df['arrest_date'])
arrests_df['semester'] = np.select(arrests_df_sem_conditions, SEMESTER_LABELS, default = "Not during a semester")
arrests_df['semester_season'] = np.select(
  [arrests_df['semester'].str.startswith("Fall"),
   arrests_df['semester'].str.startswith("Winter"),
   arrests_df['semester'].str.startswith("Spring"),
   arrests_df['semester'].str.startswith("Summer")],
  ['Fall', 'Winter', 'Spring', "Summer"],
  default = "Not during a semester"
)
arrests_df['semester_season'].value_counts()

semester_season
Spring                   1299
Fall                     1166
Summer                    755
Not during a semester     354
Winter                    154
Name: count, dtype: int64

### Splitting charges

In [96]:
arrests_df['arrested_charge']
arrests_df['arrested_charges'] = arrests_df['arrested_charge'].str.split(';')
arrests_df['arrested_charges_num'] = arrests_df['arrested_charges'].apply(lambda x: len(x))
arrests_df = arrests_df[['arrest_number', 'umpd_case_number', 'arrest_date', 'semester', 'semester_season', 'age', 'race', 'sex', 'arrested_charge', 'arrested_charges_num', 'arrested_charges']]
arrests_df_exploded = arrests_df.explode('arrested_charges')

In [100]:
arrests_df_exploded['arrested_charges_num'].describe()

count    8715.000000
mean        4.089845
std         2.813307
min         1.000000
25%         2.000000
50%         3.000000
75%         6.000000
max        15.000000
Name: arrested_charges_num, dtype: float64

In [104]:
arrests_df_exploded['arrested_charges'].value_counts(sort=True).head(10)

arrested_charges
 (Driving, Attempting to drive) veh. while under the influence of alcohol                         1101
(Driving, Attempting to drive) veh. while impaired by alcohol                                     1094
 (Driving, Attempting to drive) veh. while under the influence of alcohol per se                   660
 Reckless driving veh. in wanton and willfull disregard for safety of persons and property         483
 Negligent driving veh. in careless and imprudent manner endangering property, life and person     450
CDS:Possession Marijuana L/T 10 grams                                                              431
(Driving, Attempting to drive) veh. while under the influence of alcohol                           180
 Driver failure to obey properly placed traffic control device instructions                        142
THEFT: $100 TO UNDER $1,500-All Other                                                              121
Consuming/Possessing Open Alcoholic Beverage in Public  